# English-Luganda Translator - ML Pipeline on Colab

**GPU-Accelerated ML Workflow**

This notebook:
- Clones from GitHub
- Loads 50,000+ dataset pairs
- Trains on Tesla T4 GPU
- Calculates BLEU score

## STEP 1: Install Packages

In [ ]:
import subprocess, sys
print("="*80)
print("ENGLISH-LUGANDA TRANSLATOR - COLAB ML PIPELINE")
print("="*80)
print("\n[STEP 1: Installing packages]\n")
packages = ["torch", "transformers", "datasets", "pandas", "scikit-learn", "sacrebleu"]
print("Installing packages...")
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
    print(f"  ✓ {pkg}")
print("\n✅ Done")

## STEP 2: Clone from GitHub

In [ ]:
import os, subprocess
print("\n[STEP 2: Cloning from GitHub]\n")
if os.path.exists("/content/ENGLISH-LUGANDA-TRANSLATOR"):
    os.chdir("/content/ENGLISH-LUGANDA-TRANSLATOR")
    subprocess.run(["git", "pull"], check=False)
    print("✅ Repository updated")
else:
    subprocess.run(["git", "clone", "https://github.com/ndagirenairah/ENGLISH-LUGANDA-TRANSLATOR.git", "/content/ENGLISH-LUGANDA-TRANSLATOR"], check=True)
    os.chdir("/content/ENGLISH-LUGANDA-TRANSLATOR")
    print("✅ Repository cloned")
os.chdir("/content/ENGLISH-LUGANDA-TRANSLATOR/ENGLISH-LUGANDA-TRANSLATOR")
print(f"✅ Working: {os.getcwd()}")

## STEP 3: Load Datasets

In [ ]:
print("\n[STEP 3: Loading datasets]\n")
import sys
sys.path.insert(0, "src")
from load_data import load_all_datasets, get_dataset_statistics
combined_df = load_all_datasets()
stats = get_dataset_statistics(combined_df)
print(f"\n✅ Loaded {stats['total_samples']:,} samples")
print(f"   English avg: {stats['avg_english_length']:.1f}")
print(f"   Luganda avg: {stats['avg_luganda_length']:.1f}")

## STEP 4: Preprocess & Split

In [ ]:
print("\n[STEP 4: Preprocessing]\n")
from preprocess import preprocess_and_split, save_splits
train_df, val_df, test_df = preprocess_and_split(combined_df)
save_splits(train_df, val_df, test_df)
print(f"✅ Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

## STEP 5: Train Model (8-12 min on GPU)

In [ ]:
print("\n[STEP 5: Training]\n")
import torch, pathlib
print(f"GPU: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  Device: {torch.cuda.get_device_name(0)}")
train_file = pathlib.Path("src/train.py")
if train_file.exists():
    code = train_file.read_text()
    code = code.replace("        tokenizer=tokenizer,", "")
    train_file.write_text(code)
    print("✅ Applied hotfix\n")
print("="*80)
print("STARTING TRAINING")
print("="*80)
from train import main as train_main
try:
    model, tokenizer = train_main()
    print("\n✅ Training complete!")
except Exception as e:
    print(f"❌ Error: {e}")
    import traceback
    traceback.print_exc()

## STEP 6: Evaluate

In [ ]:
print("\n[STEP 6: Evaluating]\n")
from evaluate import main as eval_main
try:
    eval_main()
    print("\n✅ Evaluation complete!")
except Exception as e:
    print(f"❌ Error: {e}")
    import traceback
    traceback.print_exc()

## STEP 7: Results

In [ ]:
import json
from pathlib import Path
eval_file = Path("outputs/evaluation_results.json")
if eval_file.exists():
    with open(eval_file) as f:
        results = json.load(f)
    print("\n" + "="*80)
    print("FINAL RESULTS")
    print("="*80)
    print(f"\nBLEU Score: {results['bleu_score']:.2f}")
    print(f"Test samples: {results['num_test_samples']}")

✅ Pipeline Complete!